# Barkeep Protocol — Act 1.2 Training Notebook
All diagnostics saved via `run_all_diagnostics`.

## Imports

In [1]:
from dataset_wiki import make_wiki_dataset
from model import Transformer
from train import train_model, TrainConfig
from analysis import run_all_diagnostics, load_runs, plot_run_comparison

import torch

## Config

In [2]:
# ── device ──
if torch.xpu.is_available():
    DEVICE = "xpu"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

Using device: xpu


In [3]:
# ── reproducibility ──
SEED = 42
generator_global = torch.Generator(device=DEVICE).manual_seed(SEED)

In [4]:
# ── architecture hyperparameters ──
CONTEXT_SIZE   = 256
VOCAB_SIZE     = 3000
EMBEDDING_DIM  = 128
NUM_HEADS      = 4
DIM_QK         = 16   # d_k per head
DIM_V          = 32   # d_v per head
FFN_DIM        = 512
NUM_BLOCKS     = 4
DROPOUT        = 0.2
MAX_NORM       = 1
BETA_1         = 0.9
BETA_2         = 0.99
EPSILON        = 1e-8
DECAY_RATE     = 0.1

# ── training hyperparameters ──
BATCH_SIZE     = 128
STARTING_LR    = 3e-3
ENDING_LR      = 3e-4
WARMUP_STEPS   = 500
LOG_INTERVAL   = 500
NUM_ITRNS      = 5000

In [5]:
config = TrainConfig(
    log_path     = "runs.jsonl",
    lr_start     = STARTING_LR,
    lr_end       = ENDING_LR,
    iterations   = NUM_ITRNS,
    log_interval = LOG_INTERVAL,
    batch_size   = BATCH_SIZE,
    max_norm     = MAX_NORM,
    warmup_steps = WARMUP_STEPS,
    beta_1       = BETA_1,
    beta_2       = BETA_2,
    epsilon      = EPSILON,
    decay_coeff  = DECAY_RATE,
)

In [ ]:
# ── dataset ──
DATA_DIR = r"D:\Bar-Eden\Datasets\Wikitext-2"
data = make_wiki_dataset(context_size=CONTEXT_SIZE, vocab_size=VOCAB_SIZE, data_dir=DATA_DIR, device=DEVICE, seed=SEED)

trn       = data["train"]
dev       = data["dev"]
test      = data["test"]

VOCAB_SIZE = data["vocab_size"]

print(f"Vocab size: {VOCAB_SIZE}")
print("Train:", trn.inputs.shape)
print("Dev:  ", dev.inputs.shape)

In [17]:
# ── diagnostic batch (shared across all models) ──
DIAG_X = trn.inputs[:512]
DIAG_Y = trn.targets[:512]

---
## Transformer

In [ ]:
Wikitext_transformer_model = Transformer(
    CONTEXT_SIZE, VOCAB_SIZE, EMBEDDING_DIM,
    dim_qk=DIM_QK,
    dim_v=DIM_V,
    num_heads=NUM_HEADS,
    ffn_dim=FFN_DIM,
    n_blocks=NUM_BLOCKS,
    device=DEVICE, generator=generator_global, p=DROPOUT,
)
print("Optimized Transformer\n", Wikitext_transformer_model.config_dict())
print(f"Parameters:     {sum(p.nelement() for p in Wikitext_transformer_model.parameters()):,}")

In [ ]:
wikitext_transformer_results = train_model(SEED, Wikitext_transformer_model, trn, dev, config, "Wikitext Transformer", DEVICE)
torch.save(wikitext_transformer_results, "Training results/wikitext_transformer_results.pt")
torch.save(Wikitext_transformer_model, "Model Instances/wikitext_transformer.pt")

In [23]:
wikitext_transformer_model   = torch.load("Model Instances/wikitext_transformer.pt", weights_only=False)
wikitext_transformer_results = torch.load("Training results/wikitext_transformer_results.pt")

In [24]:
trn  = torch.load("Encoded Wikitext/training_set.pt", weights_only=False)
dev  = torch.load("Encoded Wikitext/dev_set.pt", weights_only=False)
test = torch.load("Encoded Wikitext/test_set.pt", weights_only=False)

In [22]:
runs = load_runs("runs.jsonl")
plot_run_comparison(runs, metric="dev_loss")
plot_run_comparison(runs, metric="train_loss")

In [19]:
run_all_diagnostics(wikitext_transformer_model, wikitext_transformer_results, DIAG_X, DIAG_Y, "Wikitext Transformer")

── Wikitext Transformer ──
  saved to D:\Bar-Eden\Act 1\Act 1.2 Transformer\Plots\Wikitext Transformer


---
## Sampling/Generating text

In [36]:
def generate(model, bpe, prompt, max_new_tokens, context_size, device,
             temperature=1.0, top_k=None):
    model.eval()
    encoded = bpe.encode(prompt)
    original_length = len(encoded)

    for _ in range(max_new_tokens):
        window = encoded[-context_size:]
        pad_len = context_size - len(window)
        padded = [0] * pad_len + window
        input_tokens = torch.tensor([padded], dtype=torch.long, device=device)

        with torch.no_grad():
            logits = model(input_tokens)[0, -1, :]   # (vocab,)

        logits = logits / temperature

        if top_k is not None:
            values, indices = torch.topk(logits, top_k)
            probs = torch.softmax(values, dim=-1)
            sampled = torch.multinomial(probs, num_samples=1)
            next_token = indices[sampled].item()
        else:
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()

        encoded.append(next_token)

    return bpe.decode(encoded[original_length:])

In [28]:
from bpe import ByteBPE

bpe = ByteBPE()
bpe.load(r"D:\Bar-Eden\Datasets\Wikitext-2\bpe.json")

In [43]:
print(generate(wikitext_transformer_model, bpe, "The history of", 400000, 256, "cpu", temperature=0.8, top_k=40))

eld . 
 A = = = 
 
 In 1941 : 00 @,@ 000 km ° 354th became a nearly 1978 @,@ 600 km south of Ireland , in Most @-@ millivehicles ( the Up @-@ south , 1st AAgadministrain ( the northern New Zealand NHill 2689 ) and the GA ) 
 Fame of the U.S. ( NSoldowe Howers = 
 
 Haifa has a result , a @-@ Dominican Republic and Dominion list @-@ Fort @-@ Civil ( Mogadishop @-@ American Hurricane Hurricane Lancashington Saint ) became the Tropical Domination Troadministration in the north @-@ east of South Australia by the California ( Roger RITC ) is the main First Eastern Gulf of Mexico Munican ( Howa ) in @-@ California LIIllas California , and the North Kingelsey ; it was the most significant loss of Lutrado ( Edward HMC ) ; however , was made into Cateinland , while Cross at its largest population in late April 1992 . 
 A tropical storm formed Cyphysis of Imptometer ( 2004 ) , at least 95 mph ) to the Mississippi ( 2015 ) . Office the Bympic Hurricane Baho = 
 
 Allosses Baja ; in Jose was born 

---